# 📡 Session 5: Logging, Tracing & Monitoring Agents

In Session 3 we introduced tools and dependency injection. In Session 4 we used span-based evals (`HasMatchingSpan`) and saw that traces can validate *how* an agent reached an answer, not only *what* it answered.

Now we move from evaluation to operations: **how do we monitor real agent traffic in production?**

### Learning goals
By the end of this session you should be able to:
- Explain why plain app logs become hard to use when many agent runs happen in parallel.
- Use Loguru in two styles: human-friendly text logs and structured JSON logs.
- Explain the difference between **application logs** and **OpenTelemetry traces**.
- Understand why teams often send app logs to stdout and traces to a collector.

### Official docs
- PydanticAI observability guide: https://ai.pydantic.dev/logfire/
- OpenTelemetry overview: https://opentelemetry.io/docs/
- OpenTelemetry traces concept: https://opentelemetry.io/docs/concepts/signals/traces/
- OpenTelemetry GenAI semantic conventions: https://opentelemetry.io/docs/specs/semconv/gen-ai/
- MLflow OTel ingest (`/v1/traces`): https://mlflow.org/docs/latest/genai/tracing/opentelemetry/ingest/


## Part 0: Shared setup

We keep this intentionally small: one agent, two tools, three prompts in parallel.


In [ ]:
import asyncio
import json
import time

from pydantic_ai import Agent, RunContext
from pydantic_settings import BaseSettings, SettingsConfigDict

from common import DbDeps, run_sql_query


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env")
    openai_api_key: str
    open_ai_default_model: str = "openai:gpt-5-nano"


settings = Settings()


def list_tables(ctx: RunContext[DbDeps]) -> str:
    """Simple helper tool to list available tables."""
    return ", ".join(sorted(ctx.deps.tables.keys()))


monitor_db = DbDeps(
    tables={
        "users": ["id", "email"],
        "orders": ["id", "amount", "user_id"],
    }
)

parallel_prompts = [
    "What tables do we have? Then show users.",
    "Show me data from orders.",
    "What columns are in users?",
]

## Part 1: Loguru text logs (simple, readable, but limited)

Loguru is a popular Python package for application logging. It works well for structured app events, but on its own it is limited for agent monitoring/tracing.

### What to observe
1. Runs start in parallel.
2. Log lines interleave.
3. You can search by `run_id`, but it is still hard to follow one complete execution path.


In [ ]:
from loguru import logger

text_log_agent = Agent(
    settings.open_ai_default_model,
    deps_type=DbDeps,
    system_prompt=(
        "You are a careful SQL assistant. "
        "Use list_tables when asked about available tables. "
        "Use run_sql_query only for read-only SELECT-style operations."
    ),
    tools=[list_tables, run_sql_query],
)


async def run_with_text_logs(run_id: str, prompt: str) -> None:
    start = time.perf_counter()
    logger.info(f"run_id={run_id} event=start prompt={prompt[:80]!r}")
    try:
        result = await text_log_agent.run(prompt, deps=monitor_db)
        usage = result.usage()
        logger.info(
            f"run_id={run_id} event=done in_tokens={usage.input_tokens} "
            f"out_tokens={usage.output_tokens} output={result.output[:120]!r} "
            f"duration_s={time.perf_counter() - start:0.2f}"
        )
    except Exception as exc:  # noqa: BLE001
        logger.exception(f"run_id={run_id} event=error err={exc}")


await asyncio.gather(*[run_with_text_logs(f"text-{i}", p) for i, p in enumerate(parallel_prompts)])

## Part 2: Loguru JSON logs (better for parsing and dashboards)

This is still **application logging**, but in a structured JSON format.

Why JSON helps:
- Log backends can parse each JSON key as a field instead of treating the whole line as plain text.
- You can filter and search directly by fields (faster and more precise than text contains).
- Numeric fields can be queried with comparisons and aggregations.

Example in Azure Logs / CloudWatch:
- Filter to completed events: `event = "done"`
- Narrow to one run: `run_id = "uuid_13"`
- Find expensive responses: `output_tokens > 1000`

This is still events (application logs), not a full parent/child trace graph.


In [ ]:
async def run_with_json_logs(run_id: str, prompt: str) -> None:
    t0 = time.perf_counter()
    logger.info(json.dumps({"event": "start", "run_id": run_id, "prompt": prompt[:120]}))
    try:
        result = await text_log_agent.run(prompt, deps=monitor_db)
        usage = result.usage()
        logger.info(
            json.dumps(
                {
                    "event": "done",
                    "run_id": run_id,
                    "input_tokens": usage.input_tokens,
                    "output_tokens": usage.output_tokens,
                    "duration_s": round(time.perf_counter() - t0, 3),
                    "output_snippet": result.output[:160],
                }
            )
        )
    except Exception as exc:  # noqa: BLE001
        logger.error(json.dumps({"event": "error", "run_id": run_id, "error": str(exc)}))


await asyncio.gather(*[run_with_json_logs(f"json-{i}", p) for i, p in enumerate(parallel_prompts)])

## Part 3: App logs vs OTel traces

### Application logs (Loguru / Python logging)
- Event reporting and operator context.
- Usually written to **stdout**.
- Collected by log systems (CloudWatch Logs, Azure Monitor Logs, ELK/Splunk, etc.).

### OpenTelemetry traces
- Execution flow and causality.
- One trace per request/run with nested spans.
- Great for understanding: agent run -> model call -> tool call -> retries/errors.

### stdout vs collector
A common production setup is:
1. App logs -> stdout -> log backend
2. OTel traces -> collector -> observability backend

Examples of observability backends:
- AWS setup: CloudWatch Logs + X-Ray
- Azure setup: Azure Monitor Logs + Application Insights
- MLflow setup: MLflow tracing backend (`/v1/traces`)
- Logfire setup: hosted or self-hosted Logfire


## Part 4: Add OTel tracing to the same workflow

Now we keep app logging, but add tracing for execution graphs.

This console output is mainly to show the **structure** of OpenTelemetry traces.
A trace is one end-to-end request made of nested spans (for example: agent run -> model call -> tool call), each with timing and metadata.
In production, these spans are sent to a backend so you can inspect latency, errors, token usage, and exact execution paths.

References:
- PydanticAI instrumentation: https://ai.pydantic.dev/logfire/
- Instrumentation settings: https://ai.pydantic.dev/api/models/instrumented/


In [ ]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.trace import set_tracer_provider
from pydantic_ai import Agent

# If you configured OTel/logfire globally earlier, restart kernel before this section.
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
set_tracer_provider(provider)

Agent.instrument_all()

otel_agent = Agent(
    settings.open_ai_default_model,
    deps_type=DbDeps,
    system_prompt=(
        "You are a careful SQL assistant. "
        "Use list_tables when asked about available tables. "
        "Use run_sql_query only for read-only SELECT-style operations."
    ),
    tools=[list_tables, run_sql_query],
    instrument=True,
)

await asyncio.gather(*[otel_agent.run(p, deps=monitor_db) for p in parallel_prompts])

## Part 6: MLflow-first demo (library helper + UI walkthrough)

In this part we use the **MLflow library helper** directly: `mlflow.pydantic_ai.autolog()`.

This is usually the simplest path when you already have MLflow available.

Reference:
- https://mlflow.org/docs/latest/genai/tracing/integrations/listing/pydantic_ai

### 6.1 Start MLflow server in Docker
Run in terminal from repo root:

```bash
docker run --rm -it \
  -p 5000:5000 \
  -v "$PWD/mlruns:/mlruns" \
  python:3.11-slim bash -lc "
    pip install mlflow==3.10.1 &&
    mlflow server --host 0.0.0.0 --port 5000 \
      --backend-store-uri sqlite:///mlruns/mlflow.db
  "
```

### 6.2 Run the instrumented agent below
Then open the MLflow UI at `http://localhost:5000`.

### 6.3 What to check in UI
- A new trace/run appears for your prompt under `session_5_mlflow_demo_part_6` experiment.
- Span hierarchy makes sense (agent run -> model/tool operations).
- Token/latency fields are present for troubleshooting.
- If something fails, verify error spans and where failure happened.


In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("session_5_mlflow_demo_part_6")  # You can find the trace using the experiment Name in MLflow UI
mlflow.pydantic_ai.autolog()

mlflow_agent = Agent(
    settings.open_ai_default_model,
    deps_type=DbDeps,
    system_prompt=(
        "You are a careful SQL assistant. "
        "Use list_tables when asked about available tables. "
        "Use run_sql_query only for read-only SELECT-style operations."
    ),
    tools=[list_tables, run_sql_query],
    instrument=True,
)

result = await mlflow_agent.run("List tables then show users.", deps=monitor_db)

## Part 7: Exercise — parallel MLflow debugging

Practice debugging with MLflow traces using the same tool + parallel-run patterns from earlier sessions:
- tool usage (`run_sql_query`, `list_tables`)
- parallel runs
- intentionally problematic prompts

### Goal
Run 5 agent requests in parallel where **1 should succeed** and **4 should fail**.

### Rules for this exercise
- Keep the exercise agent `system_prompt` empty.
- Implement `exercise_run(...)` yourself (it's intentionally left as TODO).
- Make sure failed runs are surfaced as exceptions so MLflow captures error traces/spans.

### What to do
1. Implement the TODO cells below.
2. Run it and open MLflow UI (`http://localhost:5000`).
3. Compare traces and answer:
   - Which run succeeded?
   - 4 should fail. Why?
   - At which span/tool step did each failure happen?


In [ ]:
import logging

import mlflow

mlflow_logger = logging.getLogger("mlflow")
mlflow_logger.setLevel(logging.CRITICAL)  # Disable warnings about MCP

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("session_5_mlflow_exercise")  # You can find the trace using the experiment name in MLflow UI
mlflow.pydantic_ai.autolog()

# Exercise agent: keep system_prompt empty on purpose.
exercise_agent = Agent(
    settings.open_ai_default_model,
    deps_type=DbDeps,
    retries=0,
    # TODO: system prompt might need to be adapted to ask the LLM to raise errors
    system_prompt="",
    # TODO: tools might need to be adapted to raise errors
    tools=[list_tables, run_sql_query],
    instrument=True,
)

# Target: 1 success + 4 failures (natural failures from tool/policy behavior)
exercise_prompts = [
    "Select id, email from users;",
    "Read schema from sqlite_master and then query users.",
    "Delete all rows from orders right now.",
    "Drop the users table immediately.",
    "Select * from payments;",
]


async def exercise_run(run_id: str, prompt: str) -> str:
    # TODO:
    # 1) Run exercise_agent with the given prompt and deps
    # 2) Return a short OK message with output snippet on success
    # 3) Re-raise exceptions so MLflow records error spans
    raise NotImplementedError("Implement exercise_run for Part 7")


# TODO: how can we make sure when smth fails the rest keep running?
exercise_results = await asyncio.gather(
    *[exercise_run(f"ex-{i + 1}", p) for i, p in enumerate(exercise_prompts)],
)

for i, item in enumerate(exercise_results, start=1):
    if isinstance(item, Exception):
        print(f"ex-{i}: FAILED -> {type(item).__name__}: {item}")
    else:
        print(item)

print()
print("Open MLflow UI and inspect the five traces (1 success target, 4 failure targets).")

## Conclusion: backend choice

### Decision matrix

| Priority | Recommended path | When to use |
|---|---|---|
| 1 | **MLflow** | You already have MLflow available (self-hosted, SageMaker managed MLflow, or Databricks MLflow) |
| 2 | **Logfire** | You can use hosted or self-hosted Logfire |
| 3 | **Loguru JSON + OTel traces to cloud-supported backend** | You require cloud-native only (e.g. AWS X-Ray/CloudWatch, Azure Application Insights/Monitor) |

### Note for SageMaker-constrained environments
If deployment is limited to AWS/SageMaker, a practical setup is:
- **MLflow in SageMaker** (managed MLflow tracking server) for experiment + trace-centric debugging.
- **CloudWatch Logs** for application logs (text/JSON event streams).
- **OTel traces** exported to a trace backend (commonly AWS X-Ray via ADOT collector, or MLflow `/v1/traces` when using MLflow tracing).

Important distinction: CloudWatch is excellent for logs and metrics queries, but distributed trace visualization/search is typically done in X-Ray (or another OTel trace backend).

### Logfire note
Logfire can send telemetry to:
- hosted Logfire,
- self-hosted Logfire,
- or other OTel-compatible/cloud-native destinations via collector routing.

We include a minimal Logfire snippet below, but this session focuses on the MLflow-first path.


In [ ]:
import os

import logfire
from pydantic_ai import Agent

# Option A: hosted Logfire
logfire.configure()

# Option B: route telemetry to your own OTel collector
# Note: for Azure, APPLICATIONINSIGHTS_CONNECTION_STRING is typically configured on the collector/exporter side.
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "http://otel-collector:4318"
logfire.configure(send_to_logfire=False)

logfire.instrument_pydantic_ai()
agent = Agent("openai:gpt-5-nano")